# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library. We'll follow best practices for referencing record sets, fields, and columns by their `@id`, and show dynamic ways to examine and process the dataset.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Let's load the dataset's metadata and record structure via the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (not as a dictionary)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's enumerate all available record sets in the dataset, along with their fields and columns, referencing each entity by its `@id`.

In [ ]:
# Gather all record set @id's and display structure
print("Record sets available:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for field in fields:
            print(f"    - Field @id: {field['@id']} (name: {field.get('name', '')})")
            if 'column' in field:
                columns = field['column'] if isinstance(field['column'], list) else [field['column']]
                print("      Columns:")
                for col in columns:
                    col_name = col.get('name', '')
                    print(f"        - Column @id: {col['@id']} (name: {col_name})")


## 3. Data Extraction
We will now load the primary record sets into pandas DataFrames using their `@id` references. This will make further analysis and visualization simple and consistent.

In [ ]:
# List all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

print("Record sets found:")
for rsi in record_set_ids:
    print(f"  - {rsi}")

dataframes = {}
for rsi in record_set_ids:
    # Convert records generator to DataFrame
    records = list(dataset.records(record_set=rsi))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsi] = df
        print(f"\nRecord set @id: {rsi}")
        print(f"  Fields / columns: {df.columns.tolist()}")
        print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
We will:
- Choose a numeric field (referenced by its `@id`) for filtering and normalization.
- Perform filtering, normalization, and grouping operations dynamically.
- Use record set and field @ids from the prior step for continuity.

In [ ]:
# For demonstration, pick the first record set and first numeric-like field
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Inspect the column names to select a likely numeric field (auto-select if possible)
numeric_col = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) or ('age' in col.lower() or 'interval' in col.lower()):
        numeric_col = col
        break

if not numeric_col:
    print("No obvious numeric field found! Using first column as fallback.")
    numeric_col = df.columns[0]

print(f"Using numeric field: {numeric_col}\n")

# Choose a threshold for filtering
try:
    threshold = df[numeric_col].astype(float).mean()
except:
    threshold = 0

# Filter
filtered_df = df[pd.to_numeric(df[numeric_col], errors='coerce') > threshold]
print(f"Filtered records with {numeric_col} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_col}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_col], errors='coerce') - pd.to_numeric(filtered_df[numeric_col], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_col], errors='coerce').std()
print(f"\nNormalized {numeric_col} for filtered records:")
print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

# Attempt grouping by a categorical field (look for 'sex', 'status', etc.)
group_field = None
for col in df.columns:
    if any(k in col.lower() for k in ['sex', 'status', 'location', 'type', 'msi']):
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_col].mean().to_frame()
    print(f"\nGrouped mean of {numeric_col} by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo obvious categorical field to group by.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric field and any relationships with a categorical variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(pd.to_numeric(df[numeric_col], errors='coerce').dropna(), bins=15)
plt.title(f'Distribution of {numeric_col}')
plt.xlabel(numeric_col)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_col, data=df)
    plt.title(f'{numeric_col} by {group_field}')
    plt.show()

## 6. Conclusion
In this notebook, we:
- Explored metadata and structure of a clinical oncology dataset using `mlcroissant`.
- Loaded dataframes referencing all fields and columns by their `@id`.
- Selected and processed a numeric field (by `@id`), normalized values, and optionally grouped by a relevant categorical field.
- Visualized data distributions and groupings.

This structured Croissant-based approach supports reproducible, transparent, and automatable machine learning pipeline creation for clinical and biomedical datasets.